# HunyuanWorld-1.0 — Test on Google Colab

**Перед запуском:**
1. `Runtime → Change runtime type → T4 GPU` (безкоштовно)
2. Потрібен HuggingFace токен (FLUX.1-dev — gated model):
   - Зареєструйся на https://huggingface.co
   - Прийми ліцензію: https://huggingface.co/black-forest-labs/FLUX.1-dev
   - Прийми ліцензію: https://huggingface.co/black-forest-labs/FLUX.1-Fill-dev
   - Створи токен: https://huggingface.co/settings/tokens
   - Встав токен у комірку нижче

**Очікуваний час:** ~20-40 хв (завантаження моделей + генерація)

**VRAM:** T4 = 15 GB. Використовуємо `--fp8_attention --fp8_gemm --cache` щоб вмістити.

In [ ]:
# ── Крок 0: Перевірка GPU ────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# ── Крок 1: HuggingFace логін ─────────────────────────────────────────────────
# Встав свій токен (Settings → Access Tokens → New token → Read)
HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXX"  # <-- ЗАМІНИТИ

from huggingface_hub import login
login(token=HF_TOKEN)

In [ ]:
# ── Крок 2: Клонування репо ───────────────────────────────────────────────────
import os

if not os.path.exists("HunyuanWorld-1.0"):
    !git clone https://github.com/Tencent-Hunyuan/HunyuanWorld-1.0

%cd HunyuanWorld-1.0
!ls

In [ ]:
# ── Крок 3: Залежності ────────────────────────────────────────────────────────
# Базові пакети (Colab вже має torch 2.x)
!pip install -q diffusers==0.34.0 transformers==4.51.0 accelerate sentencepiece
!pip install -q open3d trimesh kornia einops
!pip install -q huggingface_hub

# MoGe (depth estimation від Microsoft)
!pip install -q git+https://github.com/microsoft/MoGe.git

# Grounding DINO
!pip install -q groundingdino-py 2>/dev/null || \
  pip install -q git+https://github.com/IDEA-Research/GroundingDINO.git

# Real-ESRGAN
!pip install -q realesrgan basicsr

# Проект
!pip install -q -e . 2>&1 | tail -5

print("\n✅ Залежності встановлені")

In [ ]:
# ── Крок 4: Перевірка вільного місця на диску ─────────────────────────────────
!df -h /root 2>/dev/null || df -h /content
# Потрібно мінімум ~30 GB для моделей
# FLUX.1-dev: ~24 GB, FLUX.1-Fill-dev: ~24 GB (завантажуємо в bf16 = ~12 GB кожна)

In [ ]:
# ── Крок 5: Тестове зображення ───────────────────────────────────────────────
import os
os.makedirs("test_input", exist_ok=True)
os.makedirs("test_output", exist_ok=True)

# Варіант A: завантажити своє зображення
from google.colab import files
print("Завантаж зображення (або пропусти для тексту):")
# uploaded = files.upload()  # розкоментуй якщо хочеш завантажити

# Варіант B: скачати приклад із репо
if os.path.exists("examples"):
    example_imgs = []
    for root, dirs, fnames in os.walk("examples"):
        for f in fnames:
            if f.endswith(('.jpg', '.png')):
                example_imgs.append(os.path.join(root, f))
    if example_imgs:
        print(f"Знайдено приклади: {example_imgs}")
        INPUT_IMAGE = example_imgs[0]
    else:
        INPUT_IMAGE = None
else:
    INPUT_IMAGE = None

print(f"Input image: {INPUT_IMAGE}")

In [ ]:
# ── Крок 6A: Генерація панорами з ТЕКСТУ ─────────────────────────────────────
# (якщо немає вхідного зображення)

TEXT_PROMPT = "goblin camp, stone ruins, medieval fantasy, outdoor, grassland, overcast sky"
# Інші варіанти для Flare локацій:
# "dark cave dungeon, stone walls, torches, underground, fantasy"
# "ancient forest, tall trees, fantasy medieval, dirt path"
# "abandoned ruins, crumbling stone, weeds, overcast, medieval fantasy"

OUTPUT_PANO = "test_output/panorama.png"

if INPUT_IMAGE is None:
    print(f"Генерація панорами з тексту: '{TEXT_PROMPT}'")
    !python demo_panogen.py \
        --prompt "{TEXT_PROMPT}" \
        --output_path test_output \
        --fp8_attention --fp8_gemm --cache
else:
    print(f"Генерація панорами із зображення: {INPUT_IMAGE}")
    !python demo_panogen.py \
        --image_path "{INPUT_IMAGE}" \
        --output_path test_output \
        --fp8_attention --fp8_gemm --cache

In [ ]:
# ── Перевірка: показати панораму ──────────────────────────────────────────────
import os
from IPython.display import Image, display

# Знайти згенеровану панораму
pano_candidates = []
for root, dirs, files in os.walk("test_output"):
    for f in files:
        if "panorama" in f.lower() and f.endswith(".png"):
            pano_candidates.append(os.path.join(root, f))

if pano_candidates:
    pano_path = pano_candidates[0]
    print(f"Панорама: {pano_path}")
    display(Image(pano_path, width=800))
else:
    print("Панорама не знайдена. Перевір помилки вище.")
    !find test_output -type f | sort

In [ ]:
# ── Крок 7: Генерація 3D сцени з панорами ────────────────────────────────────
# labels_fg1/fg2 = що виділяти як foreground об'єкти

import glob

# Знайти панораму
pano_files = glob.glob("test_output/**/panorama.png", recursive=True) + \
             glob.glob("test_output/panorama.png")

if not pano_files:
    print("❌ Панорама не знайдена! Запусти Крок 6 спочатку.")
else:
    PANO_PATH = pano_files[0]
    print(f"Використовую панораму: {PANO_PATH}")

    # Для goblin camp:
    !python demo_scenegen.py \
        --image_path "{PANO_PATH}" \
        --labels_fg1 "stones, rocks" \
        --labels_fg2 "trees, bushes" \
        --classes outdoor \
        --output_path test_output/scene \
        --fp8_attention --fp8_gemm --cache

    print("\n✅ Генерація завершена")
    !find test_output/scene -type f | sort

In [ ]:
# ── Крок 8: Конвертація .ply → .glb (для monkey_dust_engine) ─────────────────
import trimesh
import glob
import os

os.makedirs("test_output/glb", exist_ok=True)

ply_files = glob.glob("test_output/scene/**/*.ply", recursive=True) + \
            glob.glob("test_output/scene/*.ply")

print(f"Знайдено .ply файлів: {len(ply_files)}")

converted = []
for ply_path in ply_files:
    try:
        mesh = trimesh.load(ply_path)
        name = os.path.splitext(os.path.basename(ply_path))[0]
        glb_path = f"test_output/glb/{name}.glb"
        mesh.export(glb_path)
        verts = len(mesh.vertices) if hasattr(mesh, 'vertices') else '?'
        faces = len(mesh.faces) if hasattr(mesh, 'faces') else '?'
        size_kb = os.path.getsize(glb_path) // 1024
        print(f"  ✅ {name}.ply → {name}.glb | {verts} verts, {faces} faces, {size_kb} KB")
        converted.append(glb_path)
    except Exception as e:
        print(f"  ❌ {ply_path}: {e}")

print(f"\nКонвертовано {len(converted)} файлів")

In [ ]:
# ── Крок 9: Злиття всіх шарів в один меш + статистика ────────────────────────
import trimesh
import glob
import numpy as np

ply_files = sorted(glob.glob("test_output/scene/**/*.ply", recursive=True) +
                   glob.glob("test_output/scene/*.ply"))

meshes = []
for p in ply_files:
    try:
        m = trimesh.load(p)
        if hasattr(m, 'faces'):
            meshes.append(m)
    except:
        pass

if meshes:
    combined = trimesh.util.concatenate(meshes)
    print(f"Загальна статистика:")
    print(f"  Vertices: {len(combined.vertices):,}")
    print(f"  Faces:    {len(combined.faces):,}")
    bb = combined.bounds
    print(f"  Bounds X: {bb[0][0]:.2f} .. {bb[1][0]:.2f}")
    print(f"  Bounds Y: {bb[0][1]:.2f} .. {bb[1][1]:.2f}")
    print(f"  Bounds Z: {bb[0][2]:.2f} .. {bb[1][2]:.2f}")

    # Зберегти об'єднаний
    combined.export("test_output/glb/combined_scene.glb")
    size_mb = os.path.getsize("test_output/glb/combined_scene.glb") / 1024**2
    print(f"\n✅ Збережено combined_scene.glb ({size_mb:.1f} MB)")
else:
    print("❌ Меші не знайдено")

In [ ]:
# ── Крок 10: Скачати результати ───────────────────────────────────────────────
import os
import zipfile
from google.colab import files

# Запакувати все у zip
zip_path = "hunyuan_world_result.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk("test_output"):
        for fname in filenames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, "test_output")
            zf.write(fpath, arcname)
            print(f"  + {arcname}")

size_mb = os.path.getsize(zip_path) / 1024**2
print(f"\nАрхів: {zip_path} ({size_mb:.1f} MB)")
files.download(zip_path)

## Що отримаємо у результаті

| Файл | Що це | Корисно для |
|------|-------|-------------|
| `panorama.png` | 360° рівноекутна панорама | Skybox / HDRI |
| `mesh_layer0.ply` | Background меш (земля, стіни) | Задній фон |
| `mesh_layer1.ply` | Foreground 1 (каміння тощо) | Декорації |
| `mesh_layer2.ply` | Foreground 2 (дерева тощо) | Декорації |
| `mesh_sky.ply` | Sky dome | Небо |
| `combined_scene.glb` | Всі шари разом | Перегляд у Blender |

**Обмеження виходу:**
- Vertex colors, НЕ UV-текстури
- Sheet warping = плоскі шари, не об'ємна геометрія
- Немає колізій і navmesh
- Підходить як: skybox, atmospheric backdrop, concept reference

**Для monkey_dust_engine:** відкрити `combined_scene.glb` у Blender,
перевірити якість і масштаб, за потреби — використати як skybox.